# Dynamic Hardware-Negotiated Page Rasterization

## 1. Objective
For standard vector text pages or document scans that lack raw discrete image blocks, direct high-resolution page rendering is needed. This notebook implements a dynamic backoff rasterization pipeline to securely extract absolute-edge pixel-boundary definitions (~24,000px resolution) without risking system memory crashes.

---

## 2. Advanced Algorithmic Insights

### -Negotiated Backoff Loop
Allocating massive high-resolution canvas matrices in RAM (e.g., $24,000 \times 32,000$ pixels) can easily deplete available memory, raising system allocation errors or hard-crashing Python. 
To prevent this, this notebook features an automated **Binary Step-Down Backoff Loop**:
1. It initiates page rendering at an ambitious extreme programmatic ceiling ($\approx 24,000$ pixels on the longest edge).
2. If the OS memory allocation API rejects the allocation or throws an out-of-memory exception, the pipeline handles it safely.
3. It drops the target canvas resolution by a fine-tuning interval of $500\text{px}$ and immediately attempts the allocation again.
4. It dynamically stabilizes on the **absolute peak safe limit** allowed by the hardware.

### Anti-Aliasing Killer (`set_aa_level(0)`)
Rasterization engines default to smooth out line edges using **Anti-Aliasing**, which introduces shades of gray pixels at the boundaries of characters to make them appear smoother to the human eye. 
*   While visually pleasing, this gray transition blur degrades OCR segmentation.
*   By setting `fitz.tools.set_aa_level(0)`, we force the engine to use **absolute binary pixel step-transitions**. This keeps character boundaries razor-sharp, allowing for cleaner pixel division and contour detection.

In [1]:
import fitz  # PyMuPDF
import cv2
import numpy as np
import os

def absolute_edge_dynamic_pipeline(pdf_path, output_folder="ultimate_max_pages"):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        print(f"[+] Output folder created: {output_folder}")

    # Step 1: Kill anti-aliasing to guarantee razor-sharp pixel transitions
    try:
        fitz.tools.set_aa_level(0)
        print("[+] Global anti-aliasing disabled. Forcing absolute raw pixel boundaries.")
    except Exception:
        pass

    doc = fitz.open(pdf_path)
    total_pages = len(doc)
    print(f"[+] PDF loaded successfully. Total pages: {total_pages}")
    print("[+] Strategy: Testing the ABSOLUTE EDGE of the engine (~24,000 px) with 500px fine-tuning...")
    print("="*75)

    for page_index in range(total_pages):
        page = doc.load_page(page_index)
        page_rect = page.rect
        max_page_side = max(page_rect.width, page_rect.height)
        
        # Start at a massive high-fidelity programmatic target
        target_max_side = 24_000.0 
        pix = None
        optimal_zoom = 1.0

        # Step 2: Negotiate dynamically with the engine via Backoff Loop
        while target_max_side >= 2000.0:
            optimal_zoom = target_max_side / max_page_side
            matrix = fitz.Matrix(optimal_zoom, optimal_zoom)
            try:
                # Attempt high-fidelity hardware canvas rendering
                pix = page.get_pixmap(matrix=matrix, alpha=False)
                # Break loop immediately if allocation succeeds
                break
            except Exception:
                # If engine throws memory limit error, drop 500px and re-verify the next highest ceiling
                target_max_side -= 500.0
                continue

        if pix is None:
            print(f"[!] Critical: Failed to render Page {page_index + 1} even at safe limits.")
            continue

        # Convert canvas samples directly into a multi-channel RAM memory block
        img_data = np.frombuffer(pix.samples, dtype=np.uint8).reshape((pix.h, pix.w, pix.n))
        
        # Remap color channels to match OpenCV's native BGR format without altering pixels
        if pix.n == 4:
            final_img = cv2.cvtColor(img_data, cv2.COLOR_RGBA2BGR)
        elif pix.n == 3:
            final_img = cv2.cvtColor(img_data, cv2.COLOR_RGB2BGR)
        else:
            final_img = img_data

        # Step 3: Print actual Zoom Factors and output coordinate vectors (X, Y)
        print(f"[+] Page {page_index + 1} Processed Successfully:")
        print(f"    -> Applied Zoom Factor : {optimal_zoom:.4f}x")
        print(f"    -> Image Output Vector : (X={pix.w}, Y={pix.h}) pixels")

        # Save as lossless PNG to preserve the generated 1-pixel boundary transitions
        output_image_path = os.path.join(output_folder, f"max_page_{page_index + 1}.png")
        cv2.imwrite(output_image_path, final_img)
        
        print(f"[✓] Saved successfully to: {output_image_path}")
        print("-"*75)

    print(f"\n[✓] Completed! All pages maximized to their absolute peak stable ceilings inside: {output_folder}")

if __name__ == "__main__":
    PDF_FILE_PATH = r"كشف حساب الراجحي.pdf" 
    absolute_edge_dynamic_pipeline(PDF_FILE_PATH)

[+] Output folder created: ultimate_max_pages
[+] PDF loaded successfully. Total pages: 656
[+] Strategy: Testing the ABSOLUTE EDGE of the engine (~24,000 px) with 500px fine-tuning...
[+] Page 1 Processed Successfully:
    -> Applied Zoom Factor : 26.1382x
    -> Image Output Vector : (X=15558, Y=22000) pixels
[✓] Saved successfully to: ultimate_max_pages\max_page_1.png
---------------------------------------------------------------------------
[+] Page 2 Processed Successfully:
    -> Applied Zoom Factor : 26.1382x
    -> Image Output Vector : (X=15558, Y=22000) pixels
[✓] Saved successfully to: ultimate_max_pages\max_page_2.png
---------------------------------------------------------------------------
[+] Page 3 Processed Successfully:
    -> Applied Zoom Factor : 26.1382x
    -> Image Output Vector : (X=15558, Y=22000) pixels
[✓] Saved successfully to: ultimate_max_pages\max_page_3.png
---------------------------------------------------------------------------
[+] Page 4 Processed